# MovieLens Data Exploration
## CS5100 Final Project - Data Exploration

This notebook explores the MovieLens 100K dataset, including:
- Loading data
- Basic statistics
- Data visualization
- Data quality checks

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src directory to path
sys.path.append('../src')
from data_loader import MovieLensLoader

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Imports successful!")

## 1. Load Data

In [ ]:
# Create data loader
loader = MovieLensLoader(data_path='../data/raw/ml-100k')

# Load all data
ratings, movies, users = loader.load_all()

print("\nData loading complete!")

In [ ]:
# View first few rows of ratings data
print("Sample ratings data:")
ratings.head(10)

In [ ]:
# View first few rows of movies data
print("Sample movies data:")
movies[['item_id', 'title', 'Action', 'Comedy', 'Drama']].head(10)

## 2. Dataset Statistics

In [ ]:
# Print detailed statistics
loader.print_stats()

In [ ]:
# Descriptive statistics for ratings
print("Rating distribution statistics:")
ratings['rating'].describe()

## 3. Data Visualization

### 3.1 Rating Distribution

In [ ]:
# Rating distribution histogram
plt.figure(figsize=(10, 6))
ratings['rating'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.title('Rating Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Rating', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)

# Add percentage labels
total = len(ratings)
for i, v in enumerate(ratings['rating'].value_counts().sort_index()):
    plt.text(i, v + 1000, f'{v/total*100:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../results/figures/rating_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nRating distribution:")
print(ratings['rating'].value_counts().sort_index())

### 3.2 User Activity Distribution

In [ ]:
# Number of ratings per user
ratings_per_user = ratings.groupby('user_id').size()

plt.figure(figsize=(12, 5))

# Histogram
plt.subplot(1, 2, 1)
plt.hist(ratings_per_user, bins=50, color='coral', edgecolor='black', alpha=0.7)
plt.title('User Activity Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Number of Ratings per User', fontsize=11)
plt.ylabel('Number of Users', fontsize=11)
plt.axvline(ratings_per_user.mean(), color='red', linestyle='--', 
            label=f'Mean: {ratings_per_user.mean():.1f}')
plt.legend()
plt.grid(alpha=0.3)

# Box plot
plt.subplot(1, 2, 2)
plt.boxplot(ratings_per_user, vert=True)
plt.title('User Activity Box Plot', fontsize=14, fontweight='bold')
plt.ylabel('Number of Ratings', fontsize=11)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/user_activity.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nUser activity statistics:")
print(f"Average ratings per user: {ratings_per_user.mean():.1f}")
print(f"Median: {ratings_per_user.median():.1f}")
print(f"Min: {ratings_per_user.min()}")
print(f"Max: {ratings_per_user.max()}")

### 3.3 Movie Popularity Distribution

In [ ]:
# Number of ratings per movie
ratings_per_movie = ratings.groupby('item_id').size()

plt.figure(figsize=(12, 5))

# Histogram
plt.subplot(1, 2, 1)
plt.hist(ratings_per_movie, bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
plt.title('Movie Popularity Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Number of Ratings per Movie', fontsize=11)
plt.ylabel('Number of Movies', fontsize=11)
plt.axvline(ratings_per_movie.mean(), color='red', linestyle='--',
            label=f'Mean: {ratings_per_movie.mean():.1f}')
plt.legend()
plt.grid(alpha=0.3)

# Log scale
plt.subplot(1, 2, 2)
plt.hist(ratings_per_movie, bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
plt.title('Movie Popularity (Log Scale)', fontsize=14, fontweight='bold')
plt.xlabel('Number of Ratings per Movie', fontsize=11)
plt.ylabel('Number of Movies', fontsize=11)
plt.yscale('log')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/movie_popularity.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nMovie popularity statistics:")
print(f"Average ratings per movie: {ratings_per_movie.mean():.1f}")
print(f"Median: {ratings_per_movie.median():.1f}")
print(f"Min: {ratings_per_movie.min()}")
print(f"Max: {ratings_per_movie.max()}")

# Cold start movies (fewer than 5 ratings)
cold_movies = (ratings_per_movie < 5).sum()
print(f"\nCold start movies (<5 ratings): {cold_movies} ({cold_movies/len(ratings_per_movie)*100:.1f}%)")

### 3.4 Most Popular Movies

In [ ]:
# Calculate average rating and number of ratings for each movie
movie_stats = ratings.groupby('item_id').agg({
    'rating': ['mean', 'count']
}).reset_index()
movie_stats.columns = ['item_id', 'avg_rating', 'num_ratings']

# Merge with movie titles
movie_stats = movie_stats.merge(movies[['item_id', 'title']], on='item_id')

# Only consider movies with at least 50 ratings
popular_movies = movie_stats[movie_stats['num_ratings'] >= 50]

# Top 10 highest rated movies
top_rated = popular_movies.nlargest(10, 'avg_rating')

print("\nTop 10 Highest Rated Movies (minimum 50 ratings):")
print(top_rated[['title', 'avg_rating', 'num_ratings']].to_string(index=False))

In [ ]:
# Top 10 most rated movies
most_rated = movie_stats.nlargest(10, 'num_ratings')

print("\nTop 10 Most Rated Movies:")
print(most_rated[['title', 'avg_rating', 'num_ratings']].to_string(index=False))

### 3.5 Movie Genre Distribution

In [ ]:
# Count movies in each genre
genre_columns = ['Action', 'Adventure', 'Animation', 'Children', 'Comedy',
                'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 
                'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 
                'Thriller', 'War', 'Western']

genre_counts = movies[genre_columns].sum().sort_values(ascending=True)

# Horizontal bar chart
plt.figure(figsize=(10, 8))
genre_counts.plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Movie Genre Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Number of Movies', fontsize=12)
plt.ylabel('Genre', fontsize=12)
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(genre_counts):
    plt.text(v + 5, i, str(int(v)), va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../results/figures/genre_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nGenre statistics:")
print(genre_counts.sort_values(ascending=False))

### 3.6 Data Sparsity Visualization

In [ ]:
# Create rating matrix (sample for visualization)
# Show only first 100 users and 100 movies
sample_ratings = ratings[
    (ratings['user_id'] <= 100) & 
    (ratings['item_id'] <= 100)
]

sample_matrix = sample_ratings.pivot(
    index='user_id',
    columns='item_id',
    values='rating'
)

# Visualize sparse matrix
plt.figure(figsize=(12, 10))
plt.spy(sample_matrix.notna(), markersize=1, aspect='auto')
plt.title('Rating Matrix Sparsity (First 100 users × 100 movies)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Movie ID', fontsize=11)
plt.ylabel('User ID', fontsize=11)
plt.tight_layout()
plt.savefig('../results/figures/sparsity_pattern.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nSample matrix sparsity: {sample_matrix.isna().sum().sum() / sample_matrix.size * 100:.1f}%")

## 4. Data Quality Checks

In [ ]:
print("Data quality checks:\n")

# Check for missing values
print("1. Missing value check")
print(f"   Ratings missing values: {ratings.isna().sum().sum()}")
print(f"   Movies missing values: {movies.isna().sum().sum()}")
print(f"   Users missing values: {users.isna().sum().sum()}")

# Check for duplicates
print("\n2. Duplicate check")
duplicates = ratings.duplicated(subset=['user_id', 'item_id']).sum()
print(f"   Duplicate (user, item) pairs: {duplicates}")

# Check rating range
print("\n3. Rating range check")
invalid_ratings = ((ratings['rating'] < 1) | (ratings['rating'] > 5)).sum()
print(f"   Ratings outside 1-5 range: {invalid_ratings}")

# Check ID continuity
print("\n4. ID continuity check")
missing_users = set(range(1, ratings['user_id'].max() + 1)) - set(ratings['user_id'].unique())
missing_movies = set(range(1, ratings['item_id'].max() + 1)) - set(ratings['item_id'].unique())
print(f"   Missing user IDs: {len(missing_users)}")
print(f"   Missing movie IDs: {len(missing_movies)}")

print("\nData quality check complete!")

## 5. Save Processed Data

In [ ]:
# Split into training and test sets
train_data, test_data = loader.split_train_test(test_size=0.2, random_state=42)

# Save to processed folder
train_data.to_csv('../data/processed/train_data.csv', index=False)
test_data.to_csv('../data/processed/test_data.csv', index=False)

print("\nTraining and test sets saved to data/processed/")

## Summary

Through this data exploration, we found:

1. **Dataset Size**: 100,000 ratings, 943 users, 1,682 movies
2. **Rating Distribution**: Ratings concentrated around 3-4 stars, showing normal distribution
3. **Data Sparsity**: ~93.7% sparsity, indicating most users rated only a small fraction of movies
4. **User Activity**: High variance - some users rate many movies, others very few
5. **Movie Popularity**: Long-tail distribution - few movies are very popular, most have few ratings
6. **Genre Distribution**: Drama and Comedy are the most common genres
7. **Data Quality**: No missing values, no duplicates, all ratings within valid range

These findings will help us:
- Understand baseline recommender performance
- Address the cold start problem
- Design appropriate recommendation algorithms